# M0 — Kaggle setup
Chỉ kiểm tra môi trường và smoke test. Điền commit chứa M0 sau khi push code.


In [ ]:
REPO_URL = "https://github.com/shinbkb/mini-gpt.git"
COMMIT_SHA = "REPLACE_WITH_FULL_COMMIT_CONTAINING_M0"
OFFLINE_SOURCE = None  # /kaggle/input/your-source/mini-gpt
SOURCE_COMMIT = None   # Commit nguồn của gói offline


In [ ]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys

root = Path("/kaggle/working/mini-gpt")
if OFFLINE_SOURCE:
    assert SOURCE_COMMIT and re.fullmatch(r"[0-9a-fA-F]{40}", SOURCE_COMMIT)
    if root.exists():
        raise RuntimeError("Thư mục đích đã tồn tại; dùng thư mục mới cho source offline.")
    shutil.copytree(OFFLINE_SOURCE, root,
                    ignore=shutil.ignore_patterns(".venv", "__pycache__", ".pytest_cache"))
    print("Declared offline source commit:", SOURCE_COMMIT)
else:
    assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA), "Nhập commit đầy đủ chứa M0"
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=root, text=True)
    if dirty.strip():
        raise RuntimeError("Source có thay đổi; giữ lại thay đổi trước khi checkout commit.")
    subprocess.run(["git", "fetch", "origin", COMMIT_SHA], cwd=root, check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=root, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
    assert actual.lower() == COMMIT_SHA.lower()
os.chdir(root)
if not OFFLINE_SOURCE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
subprocess.run([sys.executable, "-c", "import src; print('import src: OK')"], check=True)
subprocess.run([sys.executable, "-m", "src.environment", "--seed", "42"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
